
# Friedman Test + Post-Hoc Test with Decision Making

The **Friedman test** is a non-parametric test used to compare **three or more related/paired groups**.

Typical examples:

- The same subjects measured under 3+ conditions
- The same stocks evaluated under 3+ strategies
- The same trading days evaluated using 3+ models

This notebook demonstrates:

1. Creating example repeated-measures data
2. Running the Friedman test
3. Making a statistical decision using the p-value
4. Running a post-hoc test when Friedman is significant
5. Applying **Wilcoxon signed-rank tests with Holm correction**
6. Producing a clear decision table
7. Calculating an effect-size measure


In [ ]:

# Install packages if needed:
# %pip install numpy pandas scipy statsmodels seaborn matplotlib


In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import friedmanchisquare, wilcoxon
from statsmodels.stats.multitest import multipletests

pd.set_option("display.precision", 4)

ALPHA = 0.05



## 1. Example Data

Assume the **same 20 trading days** are evaluated using four trading strategies.

Each row is one trading day, so observations across strategies are paired.


In [ ]:

rng = np.random.default_rng(42)

n_days = 20

data = pd.DataFrame({
    "Momentum": rng.normal(1.20, 0.55, n_days),
    "Mean_Reversion": rng.normal(0.75, 0.55, n_days),
    "Breakout": rng.normal(1.00, 0.55, n_days),
    "Trend_Following": rng.normal(1.35, 0.55, n_days)
})

data.index = np.arange(1, n_days + 1)
data.index.name = "Trading_Day"

data


In [ ]:

print("Shape:", data.shape)
print("\nDescriptive statistics:")
display(data.describe().round(4))

print("\nMean return by strategy:")
display(data.mean().sort_values(ascending=False).to_frame("Mean_Return").round(4))


## 2. Visualize the paired observations

In [ ]:

plt.figure(figsize=(10, 5))

for strategy in data.columns:
    plt.plot(
        data.index,
        data[strategy],
        marker="o",
        alpha=0.7,
        label=strategy
    )

plt.axhline(0, linestyle="--")
plt.xlabel("Trading Day")
plt.ylabel("Return (%)")
plt.title("Returns of the Same Trading Days Across Strategies")
plt.legend()
plt.tight_layout()
plt.show()



## 3. Run the Friedman Test

### Hypotheses

**H₀:** The distributions/ranks of the related groups are the same.

**H₁:** At least one group differs.

### Decision rule

- **p ≤ 0.05** → Reject H₀ → statistically significant difference
- **p > 0.05** → Fail to reject H₀ → insufficient evidence of a difference

The Friedman test is appropriate here because the observations are **related/paired** and there are more than two conditions.


In [ ]:

statistic, p_value = friedmanchisquare(
    data["Momentum"],
    data["Mean_Reversion"],
    data["Breakout"],
    data["Trend_Following"]
)

print(f"Friedman statistic = {statistic:.4f}")
print(f"p-value            = {p_value:.6f}")
print(f"alpha              = {ALPHA}")


In [ ]:

if p_value <= ALPHA:
    decision = "Reject H0"
    conclusion = (
        "Statistically significant evidence that at least one "
        "related condition differs from the others."
    )
else:
    decision = "Fail to reject H0"
    conclusion = (
        "Insufficient evidence of a statistically significant "
        "difference among the related conditions."
    )

print("Decision:", decision)
print("Conclusion:", conclusion)



## 4. Post-Hoc Test

A significant Friedman test tells us that **at least one condition differs**, but it does not identify which pairs differ.

A common post-hoc approach is:

1. Perform **Wilcoxon signed-rank tests** for every pair of related groups.
2. Apply a multiple-comparison correction.
3. Here we use **Holm correction**, which controls the family-wise error rate while generally being less conservative than simple Bonferroni correction.


In [ ]:

from itertools import combinations

def friedman_posthoc_wilcoxon(
    df,
    alpha=0.05,
    correction="holm",
    zero_method="wilcox",
    alternative="two-sided"
):
    pairs = list(combinations(df.columns, 2))
    results = []

    for group1, group2 in pairs:
        x = df[group1]
        y = df[group2]

        stat, p = wilcoxon(
            x,
            y,
            zero_method=zero_method,
            alternative=alternative,
            method="auto"
        )

        results.append({
            "Group_1": group1,
            "Group_2": group2,
            "Wilcoxon_statistic": stat,
            "raw_p_value": p
        })

    result_df = pd.DataFrame(results)

    reject, corrected_p, _, _ = multipletests(
        result_df["raw_p_value"],
        alpha=alpha,
        method=correction
    )

    result_df["adjusted_p_value"] = corrected_p
    result_df["Reject_H0"] = reject

    result_df["Decision"] = np.where(
        result_df["Reject_H0"],
        "Reject H0 - Significant pair",
        "Fail to reject H0 - Not significant"
    )

    result_df["Interpretation"] = np.where(
        result_df["Reject_H0"],
        "Evidence of a difference between the paired groups",
        "Insufficient evidence of a difference"
    )

    return result_df.sort_values("adjusted_p_value").reset_index(drop=True)


In [ ]:

if p_value <= ALPHA:
    posthoc_results = friedman_posthoc_wilcoxon(
        data,
        alpha=ALPHA,
        correction="holm"
    )

    display(posthoc_results.round(6))
else:
    print(
        "Friedman test is not significant. "
        "Post-hoc pairwise testing is not required under the planned decision rule."
    )



## 5. Compact Post-Hoc Decision Table

This table makes the final decision easy to read.


In [ ]:

if p_value <= ALPHA:
    decision_table = posthoc_results[
        [
            "Group_1",
            "Group_2",
            "Wilcoxon_statistic",
            "raw_p_value",
            "adjusted_p_value",
            "Decision"
        ]
    ].copy()

    display(decision_table.round(6))



## 6. Effect Size

A statistically significant result does not tell us how large the difference is.

For the Friedman test, **Kendall's W** is commonly used as an effect-size measure.

For `n` subjects and `k` conditions:

`W = Friedman statistic / [n × (k − 1)]`

A rough interpretation is:

- Near 0 → weak agreement / small effect
- Around 0.3 → moderate effect
- Around 0.5 or higher → relatively strong effect

These cutoffs are only rough guidelines and should be interpreted in context.


In [ ]:

n = data.shape[0]
k = data.shape[1]

kendalls_w = statistic / (n * (k - 1))

print(f"Number of paired observations (n) = {n}")
print(f"Number of conditions (k)         = {k}")
print(f"Friedman statistic               = {statistic:.4f}")
print(f"Kendall's W                      = {kendalls_w:.4f}")



## 7. Complete Automated Decision Function

The function below runs the complete workflow:

**Friedman → Decision → Post-hoc Wilcoxon + Holm → Final report**


In [ ]:

def friedman_analysis(
    df,
    alpha=0.05,
    correction="holm"
):
    # Basic validation
    if df.shape[1] < 3:
        raise ValueError("Friedman test requires at least 3 related conditions.")

    if df.isna().any().any():
        raise ValueError(
            "Missing values detected. Handle missing paired observations before analysis."
        )

    # Friedman test
    values = [df[col].to_numpy() for col in df.columns]
    statistic, p_value = friedmanchisquare(*values)

    n = df.shape[0]
    k = df.shape[1]
    kendalls_w = statistic / (n * (k - 1))

    print("=" * 80)
    print("FRIEDMAN TEST DECISION REPORT")
    print("=" * 80)
    print(f"Conditions: {', '.join(df.columns)}")
    print(f"n = {n}, k = {k}")
    print(f"Friedman statistic = {statistic:.6f}")
    print(f"p-value            = {p_value:.6f}")
    print(f"alpha              = {alpha:.3f}")
    print(f"Kendall's W        = {kendalls_w:.6f}")
    print()

    if p_value <= alpha:
        print("Decision: REJECT H0")
        print(
            "Conclusion: There is statistically significant evidence "
            "that at least one related condition differs."
        )

        posthoc = friedman_posthoc_wilcoxon(
            df,
            alpha=alpha,
            correction=correction
        )

        print("\nPOST-HOC: Wilcoxon signed-rank tests with "
              f"{correction} correction")
        display(posthoc.round(6))

        return {
            "test_statistic": statistic,
            "p_value": p_value,
            "kendalls_w": kendalls_w,
            "decision": "Reject H0",
            "posthoc": posthoc
        }

    else:
        print("Decision: FAIL TO REJECT H0")
        print(
            "Conclusion: There is insufficient evidence of a "
            "statistically significant difference among the related conditions."
        )

        return {
            "test_statistic": statistic,
            "p_value": p_value,
            "kendalls_w": kendalls_w,
            "decision": "Fail to reject H0",
            "posthoc": None
        }


In [ ]:

final_results = friedman_analysis(
    data,
    alpha=ALPHA,
    correction="holm"
)



# Interpretation Template

Use the following structure when reporting results:

> A Friedman test was conducted to compare the related conditions. The test produced a Friedman statistic of **Q = ...** and **p = ...**. At α = 0.05, we **[reject / fail to reject] H₀**. Therefore, there is **[sufficient / insufficient] evidence** of a statistically significant difference among the related conditions.

If the Friedman test is significant:

> Post-hoc Wilcoxon signed-rank tests with Holm correction were conducted. The significant pairs were **...**, while the remaining comparisons were not statistically significant after multiple-comparison correction.

Also report **Kendall's W** when useful to communicate the magnitude of the overall effect.



# Important Assumptions and Practical Notes

### Friedman test is appropriate when:

- The dependent variable is at least ordinal.
- There are **3 or more related/paired conditions**.
- The same subjects/units are measured under each condition.
- Observations across subjects/units are independent.
- The response can reasonably be ranked within each subject/unit.

### Example

**Correct:**

| Day | Strategy A | Strategy B | Strategy C |
|---|---:|---:|---:|
| 1 | 1.2 | 0.8 | 1.5 |
| 2 | 0.5 | 0.9 | 1.1 |
| 3 | 1.4 | 0.7 | 1.2 |

The same trading days are evaluated under every strategy, so the observations are paired.

### Not the same as one-way ANOVA

- **One-way ANOVA:** independent groups, parametric assumptions.
- **Kruskal-Wallis:** 3+ independent groups, non-parametric.
- **Friedman:** 3+ related/paired groups, non-parametric.
- **Wilcoxon signed-rank:** two related/paired groups.

### Multiple comparisons matter

If the Friedman test is significant, do not run many pairwise tests without correction. Holm, Bonferroni, or another appropriate correction should be used to control the family-wise error rate.
